# Zimba Town Council: master cleaning notebook

CSC 4792 - Group 45

This notebook combines all content from the three individual cleaning notebooks,
in the order below. Original markdown and code cells are retained, including
dataset-specific Detect -> Judge -> Act decisions and handoff summaries.

1. [CDF projects and source posts](#cdf-section)
2. [Council administration](#admin-section)
3. [District profile](#district-section)

Run from the repository root with pandas installed (`python -m pip install -r
requirements.txt`). Run cells in order. The export cells write the cleaned CSVs
in `data/`; each section documents its preserved input snapshot. No network
access is needed. Variables may be reused between sections, so run each section
from its setup cell when working on it independently.

Scope: this combines the existing cleaning notebooks; it does not add scraping
code, a data-description paper, or other material absent from those notebooks.


<a id="cdf-section"></a>

# CDF projects and source posts

Source: `cdf_cleaning.ipynb`


# CDF datasets: Part 1 pandas cleaning
This notebook implements sections 1.1-1.8 of the supplied cleaning instructions for CDF projects and source posts only. The manually reviewed extraction snapshot is the input, not a substitute for cleaning. Every cleaning operation below uses pandas; original snapshots and narrative evidence are preserved.

Run from the repository root. No network access is needed. The final cells export the existing submission filenames. This notebook covers only the CDF contribution, not the group's complete Part 2 notebook.


In [1]:
from pathlib import Path
import re
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "scripts/clean_cdf.py").exists(), "Run this notebook from the repository root"
PROJECT_FILE = "db-unza26-csc4792-zimba_town_council_cdf_projects.csv"
SOURCE_FILE = "cdf_source_posts.csv"
snapshot = ROOT / "raw/cdf_projects/before_cleaning"
cdf = pd.read_csv(snapshot / PROJECT_FILE, sep="|")
posts = pd.read_csv(snapshot / SOURCE_FILE, sep="|")
for name, df in [("cdf", cdf), ("posts", posts)]:
    print(name, df.shape)
    print(df.head().to_string(index=False))
    df.info()
    print(df.describe(include="all").to_string())


cdf (44, 10)
project_id                              project_name    sector constituency funding_source  funding_amount_zmw    status date_reported                                                                                                                                             description                             source_url
   ZTC-001    2023 CDF cooperative and company loans     other    Mapatizya            CDF                 NaN completed    2023-11-02             Loans were disbursed to 26 cooperatives and companies. The amount is left blank because the source prints the malformed figure K2, 93, 400. https://www.zimbacouncil.gov.zm/?p=911
   ZTC-002      2023 CDF women and youth club grants     other    Mapatizya            CDF           2143990.0 completed    2023-11-02                        K2,143,990 in grants was awarded to 65 women and youth clubs. Completed refers to the reported award, not subsequent activities. https://www.zimbacouncil.gov.zm/?p=911
   ZTC-00

## 1.2 Duplicates: Detect
Count exact duplicates and repeated source URLs, and display the projects sharing URLs. A repeated URL is a candidate for investigation, not proof of duplicated project data.


In [2]:
print("Exact project duplicates:", cdf.duplicated().sum())
print("Repeated project URLs:", cdf.duplicated(subset=["source_url"]).sum())
print("Exact source-post duplicates:", posts.duplicated().sum())
print("Repeated source-post URLs:", posts.duplicated(subset=["url"]).sum())
print(cdf.loc[cdf.duplicated(subset=["source_url"], keep=False),
              ["project_id", "project_name", "source_url"]].to_string(index=False))


Exact project duplicates: 0
Repeated project URLs: 25
Exact source-post duplicates: 0
Repeated source-post URLs: 0
project_id                                                         project_name                              source_url
   ZTC-001                               2023 CDF cooperative and company loans  https://www.zimbacouncil.gov.zm/?p=911
   ZTC-002                                 2023 CDF women and youth club grants  https://www.zimbacouncil.gov.zm/?p=911
   ZTC-003                             Mapatizya 2620 school desks distribution  https://www.zimbacouncil.gov.zm/?p=911
   ZTC-004                            Cikuyu Primary School 1x3 classroom block  https://www.zimbacouncil.gov.zm/?p=982
   ZTC-005                                                   Mbwiko Ward clinic  https://www.zimbacouncil.gov.zm/?p=982
   ZTC-006                                                 Mulamfwu Ward clinic  https://www.zimbacouncil.gov.zm/?p=982
   ZTC-007                            Muziya 

## 1.2 Duplicates: Judge and Act
The 25 repeated project URLs refer to different projects bundled in 19 articles. URL-only deduplication would delete valid records, so the PDF example's one-project-per-article assumption does not apply. Retain distinct projects and separate-date updates; use all original fields except the sequential ID to remove only true project duplicates. Source articles are deduplicated by URL only after confirming no URL has conflicting contents.


In [3]:
project_keys = [c for c in cdf.columns if c != "project_id"]
cdf_clean = cdf.drop_duplicates(subset=project_keys, keep="first").copy()
source_variants = posts.drop_duplicates()
assert not source_variants.duplicated(subset=["url"]).any(), "Review conflicting source versions first"
posts_clean = posts.drop_duplicates(subset=["url"], keep="first").copy()
print("Project duplicates removed:", len(cdf) - len(cdf_clean))
print("Source duplicates removed:", len(posts) - len(posts_clean))


Project duplicates removed: 0
Source duplicates removed: 0


## 1.3 Missing values: Detect
Use `.isnull().sum()` before deciding whether gaps make a record unusable.


In [4]:
print("Project missing values:\n", cdf_clean.isnull().sum())
print("Source missing values:\n", posts_clean.isnull().sum())


Project missing values:
 project_id             0
project_name           0
sector                 0
constituency           2
funding_source         0
funding_amount_zmw    33
status                 4
date_reported          0
description            0
source_url             0
dtype: int64
Source missing values:
 title        0
date         0
body_text    0
url          0
dtype: int64


## 1.3 Missing values: Judge and Act
Funding and status may be targets in later analyses, but useful records remain usable for other questions. Follow the PDF's explicit action table: leave 33 unknown amounts blank, label four unstated statuses `unspecified`, and retain two missing locations without guessing. The illustrative `dropna(status)` snippet is not appropriate here. Description and body text are supporting features and remain unchanged; no rows are dropped for missing targets in this snapshot.


In [5]:
cdf_clean["status"] = cdf_clean["status"].astype("string").str.strip().str.lower().replace("", pd.NA).fillna("unspecified")
# Keep amounts as numeric missing values during analysis, exporting them as blank cells.
funding = pd.to_numeric(cdf_clean["funding_amount_zmw"], errors="coerce")
assert not (cdf_clean["funding_amount_zmw"].notna() & funding.isna()).any(), "Review invalid amounts"
cdf_clean["funding_amount_zmw"] = funding
print(cdf_clean.isnull().sum())
print(cdf_clean["status"].value_counts(dropna=False))


project_id             0
project_name           0
sector                 0
constituency           2
funding_source         0
funding_amount_zmw    33
status                 0
date_reported          0
description            0
source_url             0
dtype: int64
status
planned            20
completed          17
unspecified         4
ongoing             2
near_completion     1
Name: count, dtype: Int64


## 1.4 Outliers: Detect
Compute quartiles and 1.5-IQR bounds on observed funding amounts. Dates and numbers embedded in article text are not measurement columns. Repeated announcements mean these amounts must not be summed as total expenditure.


In [6]:
Q1 = cdf_clean["funding_amount_zmw"].quantile(0.25)
Q3 = cdf_clean["funding_amount_zmw"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = cdf_clean[(cdf_clean["funding_amount_zmw"] < lower) | (cdf_clean["funding_amount_zmw"] > upper)]
print({"Q1": Q1, "Q3": Q3, "IQR": IQR, "lower": lower, "upper": upper})
print(outliers[["project_name", "funding_amount_zmw", "source_url"]].to_string(index=False))


{'Q1': np.float64(700000.0), 'Q3': np.float64(1922924.31), 'IQR': np.float64(1222924.31), 'lower': np.float64(-1134386.465), 'upper': np.float64(3757310.7750000004)}
Empty DataFrame
Columns: [project_name, funding_amount_zmw, source_url]
Index: []


## 1.4 Outliers: Judge and Act
No values are flagged in this snapshot, so no source investigations or corrections are necessary. Keep all observed values. If a later dataset produces flags, stop to review their saved source HTML/URLs; do not delete an extreme value simply because it is large.


In [7]:
assert outliers.empty, "Review flagged amounts against their sources before continuing"
assert (cdf_clean["funding_amount_zmw"].dropna() > 0).all()
print("No outliers removed or corrected.")


No outliers removed or corrected.


## 1.5 Optional text cleaning: Judge and Act
Preserve original narrative evidence and add separate analysis features. Remove HTML first, then case-fold, remove punctuation through word tokenization and remove an explicit small stopword list. Regex tokenization uses the allowed toolkit without NLTK downloads; negations such as `not` are retained. This is a documented regex equivalent of the optional example, not a claim to use NLTK's tokenizer or full English stopword corpus.


In [8]:
stop_words = set("a an the and or of to in on at for from by with as is are was were be been being it its this that these those".split())
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"<[^>]+>", " ", str(text))
    text = text.casefold()
    tokens = re.findall(r"\b\w+\b", text)
    return " ".join(token for token in tokens if token not in stop_words)

cdf_clean["description_clean"] = cdf_clean["description"].apply(clean_text)
posts_clean["body_text_clean"] = posts_clean["body_text"].apply(clean_text)
print(cdf_clean[["description", "description_clean"]].head().to_string(index=False))


                                                                                                                                            description                                                                                                           description_clean
            Loans were disbursed to 26 cooperatives and companies. The amount is left blank because the source prints the malformed figure K2, 93, 400.                loans disbursed 26 cooperatives companies amount left blank because source prints malformed figure k2 93 400
                       K2,143,990 in grants was awarded to 65 women and youth clubs. Completed refers to the reported award, not subsequent activities.                    k2 143 990 grants awarded 65 women youth clubs completed refers reported award not subsequent activities
Distribution of 2,620 desks procured under 2022 and 2023 CDF was launched. The printed amount K 2, 887,051 is normalized by removing spaces and commas. distribution 2 620 d

## 1.6 Standardize types and categories
Convert dates with `pd.to_datetime`, funding with `pd.to_numeric`, and strip/lowercase categories. Stop on invalid nonblank values rather than silently lose evidence. CSV files do not store pandas dtypes, so dates serialize in ISO format and numeric gaps serialize as blanks.


In [9]:
for frame, column in [(cdf_clean, "date_reported"), (posts_clean, "date")]:
    parsed = pd.to_datetime(frame[column], errors="coerce")
    assert not (frame[column].notna() & parsed.isna()).any(), "Review invalid dates"
    frame[column] = parsed
cdf_clean["funding_amount_zmw"] = pd.to_numeric(cdf_clean["funding_amount_zmw"], errors="coerce")
for column in ["status", "sector"]:
    cdf_clean[column] = cdf_clean[column].astype("string").str.strip().str.lower()
assert cdf_clean["status"].isin(["planned", "ongoing", "near_completion", "completed", "unspecified"]).all()
assert cdf_clean["sector"].isin(["education", "health", "water_sanitation", "agriculture", "infrastructure", "other"]).all()
assert cdf_clean["source_url"].notna().all() and posts_clean["url"].notna().all()
for name, df in [("cdf_clean", cdf_clean), ("posts_clean", posts_clean)]:
    print(name, df.shape)
    df.info()
    print(df.describe(include="all").to_string())


cdf_clean (44, 11)
<class 'pandas.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   project_id          44 non-null     str           
 1   project_name        44 non-null     str           
 2   sector              44 non-null     string        
 3   constituency        42 non-null     str           
 4   funding_source      44 non-null     str           
 5   funding_amount_zmw  11 non-null     float64       
 6   status              44 non-null     string        
 7   date_reported       44 non-null     datetime64[us]
 8   description         44 non-null     str           
 9   source_url          44 non-null     str           
 10  description_clean   44 non-null     str           
dtypes: datetime64[us](1), float64(1), str(7), string(2)
memory usage: 3.9 KB
       project_id                            project_name     sector constituency f

## 1.7 Export and verify
Export only the two CDF submission files, with their original filenames, pipe separators and no index. Reload both exports to validate their dimensions and columns. Compare this explicitly implemented notebook pipeline against the standalone pandas cleaner to prevent divergent outputs.


In [10]:
import importlib.util
spec = importlib.util.spec_from_file_location("cdf_cleaning_script", ROOT / "scripts/clean_cdf.py")
cleaner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cleaner)
expected_cdf, _ = cleaner.clean(cdf, True)
expected_posts, _ = cleaner.clean(posts, False)
pd.testing.assert_frame_equal(cdf_clean, expected_cdf)
pd.testing.assert_frame_equal(posts_clean, expected_posts)
for filename, df in [(PROJECT_FILE, cdf_clean), (SOURCE_FILE, posts_clean)]:
    target = ROOT / "data" / filename
    exported = df.to_csv(sep="|", index=False, date_format="%Y-%m-%d", na_rep="").encode("utf-8-sig")
    # Do not rewrite identical exports, which may be open in a spreadsheet app.
    if not target.exists() or target.read_bytes() != exported:
        target.write_bytes(exported)
    reopened = pd.read_csv(target, sep="|")
    assert reopened.shape == df.shape
    assert list(reopened.columns) == list(df.columns)
    assert "|" in target.read_text(encoding="utf-8-sig").splitlines()[0]
    print(filename, reopened.shape)
print("Notebook and standalone pandas pipeline agree.")


db-unza26-csc4792-zimba_town_council_cdf_projects.csv (44, 11)
cdf_source_posts.csv (29, 5)
Notebook and standalone pandas pipeline agree.


## 1.8 Handoff
**Projects:** 44 announcement rows and 11 columns; no true duplicates removed. Unknown amounts remain blank in 33 rows and locations in two; four unstated statuses are `unspecified`. No IQR outliers were flagged or removed. Original descriptions accompany the new `description_clean` feature.

**Source posts:** 29 article rows and five columns; no duplicates removed and no missing original fields. Dates are standardized and original narrative text accompanies `body_text_clean`. Numeric IQR analysis does not apply to this text evidence table. Export round trips and agreement with the standalone pandas pipeline are checked above.

Supporting evidence: `raw/cdf_projects/before_cleaning/`, `raw/cdf_projects/post_*.html`, and `docs/cdf_cleaning/README.md`. This preserves legitimate multi-project articles and is not a unique-project expenditure register.


<a id="admin-section"></a>

# Council administration

Source: `council_admin_cleaning.ipynb`


# Council administration cleaning: Detect -> Judge -> Act
This notebook uses pandas and regex to inspect, clean and export only the council administration dataset. Run from the repository root. Input is preserved on first execution under `raw/before_cleaning/`; later runs replay that snapshot. No web requests or inferred demographic/contact values are introduced. The CDF datasets are not modified.


In [11]:
from pathlib import Path
import html
import re
import shutil
import pandas as pd

ROOT = Path.cwd()
filename = "db-unza26-csc4792-zimba_town_council_admin.csv"
target = ROOT / "data" / filename
snapshot = ROOT / "raw/before_cleaning" / filename
assert target.exists(), "Run from the repository root"
snapshot.parent.mkdir(parents=True, exist_ok=True)
if not snapshot.exists():
    shutil.copyfile(target, snapshot)
# Read as strings so phone prefixes and identifiers survive loading.
raw = pd.read_csv(snapshot, sep="|", dtype="string")
print("Input shape:", raw.shape)
print(raw.head().to_string(index=False))
raw.info()
print(raw.describe(include="all").to_string())
print("Missing values:\n", raw.isnull().sum())


Input shape: (2, 9)
  record_id name role_title department      phone                          email office_address                     source_page                                   source_url
ZTC-ADM-001 <NA>       <NA>       <NA>       <NA>        zimbacouncil@grz.gov.zm           <NA>      Zimba Town Council – Zimba             https://www.zimbacouncil.gov.zm/
ZTC-ADM-002 <NA>       <NA>       <NA> 0978080795 www.zimbatowncouncil@gmail.com           <NA> Contact Us – Zimba Town Council https://www.zimbacouncil.gov.zm/?page_id=275
<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   record_id       2 non-null      string
 1   name            0 non-null      string
 2   role_title      0 non-null      string
 3   department      0 non-null      string
 4   phone           1 non-null      string
 5   email           2 non-null      string
 6   office_address 

## Duplicates and text normalization: Detect
Inspect exact duplicates and repeated source URLs before making removal decisions. Decode HTML entities and collapse whitespace in ordinary text. Preserve URL punctuation, telephone prefixes and names; these are evidence fields, not a bag of words.


In [12]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Repeated source URLs:", raw.duplicated(subset=["source_url"]).sum())
clean = raw.copy()
def normalize(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", html.unescape(str(value))).strip() or pd.NA
for column in clean.columns:
    if column.endswith("url"):
        clean[column] = clean[column].str.strip().replace("", pd.NA)
    else:
        clean[column] = clean[column].map(normalize).astype("string")


Exact duplicate rows: 0
Repeated source URLs: 0


## Duplicates: Judge and Act
The current file has two records from different contact pages. Remove exact duplicates excluding record IDs; only deduplicate by URL after confirming remaining rows on that URL have identical contents. Different officials or departments may share a page, so conflicting same-URL rows require review instead of silently keeping the first. This gives the same result as the existing administration cleaner on the current data while protecting future multi-record pages.


In [13]:
before = len(clean)
clean = clean.drop_duplicates(subset=[c for c in clean.columns if c != "record_id"], keep="first").copy()
assert not clean.duplicated(subset=["source_url"]).any(), "Review distinct records sharing a page"
clean = clean.drop_duplicates(subset=["source_url"], keep="first").copy()
duplicates_removed = before - len(clean)
print("Duplicates removed:", duplicates_removed)


Duplicates removed: 0


## Missing values: Detect -> Judge -> Act
This contact directory has no predefined prediction target. Names, role titles, departments, telephone numbers, emails and addresses are supporting features; do not invent a person's identity or discard a usable email-only contact point. Missing source URLs make a record unauditable and require review. Retain other gaps as blank cells. Keep phone values as strings, including the leading zero. Lowercase emails and normalize phone whitespace without guessing country codes or altering published digits.

IQR is not applicable: this file contains no numeric measurement columns. Phone numbers and record IDs are identifiers, not quantities; outliers corrected/removed are therefore not applicable rather than evidence of an IQR test.


In [14]:
print("Missing values:\n", clean.isnull().sum())
clean["email"] = clean["email"].str.lower()
clean["phone"] = clean["phone"].str.replace(r"\s+", " ", regex=True).str.strip()
email_pattern = r"[^\s@]+@[^\s@]+\.[^\s@]+"
invalid_email = clean["email"].notna() & ~clean["email"].str.fullmatch(email_pattern, na=False)
print("Email syntax flags:", int(invalid_email.sum()))
assert not invalid_email.any(), "Review email syntax without guessing a replacement"
print("IQR: not applicable; no numeric measurements")


Missing values:
 record_id         0
name              2
role_title        2
department        2
phone             1
email             0
office_address    2
source_page       0
source_url        0
dtype: int64
Email syntax flags: 0
IQR: not applicable; no numeric measurements


## Final validation and export
Preserve the original columns and source links. Optional punctuation removal/tokenization is omitted for this structured directory/profile: it would damage contacts and is unnecessary for the demographic measurements. Narrative notes remain intact. Export with `sep='|'` and `index=False`, retaining blanks, then reload as strings to check every exported field and schema. Existing identical files are not rewritten, allowing a file to remain open in a spreadsheet application.


In [15]:
assert clean["source_url"].notna().all(), "Review missing traceability URLs"
assert clean["source_url"].str.match(r"https?://", na=False).all()
assert clean["record_id"].notna().all() and clean["record_id"].is_unique
assert list(clean.columns) == list(raw.columns)
clean.info()
print(clean.describe(include="all").to_string())
exported = clean.to_csv(sep="|", index=False, na_rep="").encode("utf-8-sig")
if target.read_bytes() != exported:
    target.write_bytes(exported)
roundtrip = pd.read_csv(target, sep="|", dtype="string")
assert roundtrip.shape == clean.shape
pd.testing.assert_frame_equal(roundtrip.fillna(""), clean.astype("string").fillna(""), check_dtype=False)
print("Final rows:", len(clean), "columns:", len(clean.columns))
print("Duplicates removed:", duplicates_removed)
print("Final missing values:\n", clean.isnull().sum())
print("Saved:", filename)


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   record_id       2 non-null      string
 1   name            0 non-null      string
 2   role_title      0 non-null      string
 3   department      0 non-null      string
 4   phone           1 non-null      string
 5   email           2 non-null      string
 6   office_address  0 non-null      string
 7   source_page     2 non-null      string
 8   source_url      2 non-null      string
dtypes: string(9)
memory usage: 276.0 bytes
          record_id name role_title department       phone                    email office_address                 source_page                        source_url
count             2    0          0          0           1                        2              0                           2                                 2
unique            2    0          0          0           1   

## Handoff summary
The administration directory retains two rows and nine columns, with no duplicates removed. Missing names, roles, departments and addresses remain blank in both records, while one telephone number is missing; both records retain emails and source URLs. Text whitespace/entities and email case are normalized while telephone digits and prefixes remain unchanged. Numeric outlier analysis is not applicable to this contact directory.


<a id="district-section"></a>

# District profile

Source: `district_profile_cleaning.ipynb`


# District profile cleaning: Detect -> Judge -> Act
This notebook uses pandas and regex to inspect, clean and export only the district profile dataset. Run from the repository root. Input is preserved on first execution under `raw/before_cleaning/`; later runs replay that snapshot. No web requests or inferred demographic/contact values are introduced. The CDF datasets are not modified.


In [16]:
from pathlib import Path
import html
import re
import shutil
import pandas as pd

ROOT = Path.cwd()
filename = "db-unza26-csc4792-zimba_town_council_district_profile.csv"
target = ROOT / "data" / filename
snapshot = ROOT / "raw/before_cleaning" / filename
assert target.exists(), "Run from the repository root"
snapshot.parent.mkdir(parents=True, exist_ok=True)
if not snapshot.exists():
    shutil.copyfile(target, snapshot)
# Read as strings so phone prefixes and identifiers survive loading.
raw = pd.read_csv(snapshot, sep="|", dtype="string")
print("Input shape:", raw.shape)
print(raw.head().to_string(index=False))
raw.info()
print(raw.describe(include="all").to_string())
print("Missing values:\n", raw.isnull().sum())


Input shape: (4, 14)
   record_id        level      name population population_year population_male population_female households area_km2 province             neighboring_districts                                                                                                                                                                                                                 notes                                    source_url                         secondary_source_url
ZTC-DIST-001     district     Zimba      66725            2010           32186             34539      13284     5245 Southern Kalomo;Kazungula;Choma;Sinazongwe                                                                                                                   2010 Census by the Central Statistical Office (CSO); annual population growth rate reported as 2.9%  https://www.zimbacouncil.gov.zm/?page_id=759                                         <NA>
ZTC-DIST-002     district     Zimba      98533   

## Duplicates and text normalization: Detect
Inspect exact duplicates and repeated source URLs before making removal decisions. Decode HTML entities and collapse whitespace in ordinary text. Preserve URL punctuation, telephone prefixes and names; these are evidence fields, not a bag of words.


In [17]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Repeated source URLs:", raw.duplicated(subset=["source_url"]).sum())
clean = raw.copy()
def normalize(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", html.unescape(str(value))).strip() or pd.NA
for column in clean.columns:
    if column.endswith("url"):
        clean[column] = clean[column].str.strip().replace("", pd.NA)
    else:
        clean[column] = clean[column].map(normalize).astype("string")


Exact duplicate rows: 0
Repeated source URLs: 2


## Duplicates: Judge and Act
Three Zimba rows report 2010 census data, a 2018 projection and 2022 census data. Neither a shared URL nor name/level alone makes these duplicates. Remove only identical non-ID records and check name/level/year for conflicting observations. Preserve projection notes and report years so the observations are not misrepresented as equivalent census counts.


In [18]:
clean["level"] = clean["level"].str.lower().str.strip()
keys = [column for column in clean.columns if column != "record_id"]
before = len(clean)
clean = clean.drop_duplicates(subset=keys, keep="first").copy()
duplicates_removed = before - len(clean)
conflicts = clean[clean.duplicated(subset=["name", "level", "population_year"], keep=False)]
print("True duplicates removed:", duplicates_removed)
print("Conflicting name/level/year records:", len(conflicts))
assert conflicts.empty, "Review conflicting observations before export"


True duplicates removed: 0
Conflicting name/level/year records: 0


## Missing values and types: Detect -> Judge -> Act
Population and area are potential targets for demographic analyses; year is essential context for a population observation. Male/female counts and household counts are numeric supporting features or targets for other questions. Missing values are retained because the published constituency row is still useful as an administrative reference, and later-year district rows remain useful without sex/household breakdowns. Never copy a district total into a constituency row or carry 2010 counts forward to 2022. Province, neighbors, notes and secondary URLs are supporting features. A missing secondary URL means no secondary source is recorded, not a missing primary source.

Use `pd.to_numeric(errors='coerce')` and check whether conversion would discard a nonblank input. No outside-source supplementation is claimed. Missing demographic figures remain an explicitly documented limitation.


In [19]:
print(clean.isnull().sum())
numeric = ["population", "population_year", "population_male", "population_female", "households", "area_km2"]
for column in numeric:
    original = clean[column]
    values = pd.to_numeric(original, errors="coerce")
    assert not (original.notna() & values.isna()).any(), f"Review invalid {column}"
    assert (values.dropna() >= 0).all(), f"Review negative {column}"
    if column != "area_km2":
        assert (values.dropna() % 1 == 0).all(), f"Review fractional {column}"
        clean[column] = values.astype("Int64")
    else:
        clean[column] = values.astype("Float64")
assert clean.loc[clean["population"].notna(), "population_year"].notna().all()
assert clean["level"].isin(["district", "constituency", "ward"]).all()
both = clean[["population", "population_male", "population_female"]].notna().all(axis=1)
assert (clean.loc[both, "population_male"] + clean.loc[both, "population_female"] == clean.loc[both, "population"]).all()


record_id                0
level                    0
name                     0
population               1
population_year          1
population_male          3
population_female        3
households               3
area_km2                 1
province                 0
neighboring_districts    1
notes                    0
source_url               0
secondary_source_url     3
dtype: int64


## Outliers: Detect -> Judge -> Act
Compute 1.5-IQR fences separately by administrative level, excluding missing measurements and year identifiers. With only three district population observations, one sex/household observation and a repeated district area, these diagnostics are weak: no flags does not establish accuracy. Do not compare constituency and district populations as if they were the same measurement scale. A future flagged value requires checking its source; this notebook stops rather than removing or correcting it automatically.


In [20]:
iqr_rows = []
flagged_indices = set()
for level, group in clean.groupby("level"):
    for column in ["population", "population_male", "population_female", "households", "area_km2"]:
        values = group[column].dropna().astype(float)
        if values.empty:
            continue
        Q1, Q3 = values.quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        flags = values[(values < lower) | (values > upper)]
        flagged_indices.update(flags.index)
        iqr_rows.append({"level": level, "column": column, "n": len(values), "lower": lower, "upper": upper, "flags": len(flags)})
print(pd.DataFrame(iqr_rows).to_string(index=False))
print("Flagged records:", len(flagged_indices))
assert not flagged_indices, "Review source evidence for flagged values before export"
outliers_removed = outliers_corrected = 0


   level            column  n    lower     upper  flags
district        population  3 51435.75 134617.75      0
district   population_male  1 32186.00  32186.00      0
district population_female  1 34539.00  34539.00      0
district        households  1 13284.00  13284.00      0
district          area_km2  3  5245.00   5245.00      0
Flagged records: 0


## Final validation and export
Preserve the original columns and source links. Optional punctuation removal/tokenization is omitted for this structured directory/profile: it would damage contacts and is unnecessary for the demographic measurements. Narrative notes remain intact. Export with `sep='|'` and `index=False`, retaining blanks, then reload as strings to check every exported field and schema. Existing identical files are not rewritten, allowing a file to remain open in a spreadsheet application.


In [21]:
assert clean["source_url"].notna().all(), "Review missing traceability URLs"
assert clean["source_url"].str.match(r"https?://", na=False).all()
assert clean["record_id"].notna().all() and clean["record_id"].is_unique
assert list(clean.columns) == list(raw.columns)
clean.info()
print(clean.describe(include="all").to_string())
exported = clean.to_csv(sep="|", index=False, na_rep="").encode("utf-8-sig")
if target.read_bytes() != exported:
    target.write_bytes(exported)
roundtrip = pd.read_csv(target, sep="|", dtype="string")
assert roundtrip.shape == clean.shape
pd.testing.assert_frame_equal(roundtrip.fillna(""), clean.astype("string").fillna(""), check_dtype=False)
print("Final rows:", len(clean), "columns:", len(clean.columns))
print("Duplicates removed:", duplicates_removed)
print("Final missing values:\n", clean.isnull().sum())
print("Saved:", filename)


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   record_id              4 non-null      string 
 1   level                  4 non-null      string 
 2   name                   4 non-null      string 
 3   population             3 non-null      Int64  
 4   population_year        3 non-null      Int64  
 5   population_male        1 non-null      Int64  
 6   population_female      1 non-null      Int64  
 7   households             1 non-null      Int64  
 8   area_km2               3 non-null      Float64
 9   province               4 non-null      string 
 10  neighboring_districts  3 non-null      string 
 11  notes                  4 non-null      string 
 12  source_url             4 non-null      string 
 13  secondary_source_url   1 non-null      string 
dtypes: Float64(1), Int64(5), string(8)
memory usage: 604.0 bytes
           r

## Handoff summary
The district profile retains four rows and 14 columns, including the separate 2010 census, 2018 projection and 2022 census observations. No true duplicates are removed, and no IQR outliers are corrected or removed; the small sample limits outlier inference. Missing constituency measurements and later-year sex/household breakdowns remain blank rather than estimated. Notes and source URLs are preserved, and integer measurements use nullable integer types during processing.
